[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kartikmunjal/rlhf-and-reward-modelling/blob/main/notebooks/03_reward_modeling.ipynb)

# Notebook 3 — Stage 2: Reward Modeling

## The Core Problem

RL requires a *reward signal*.  
We can't have humans in the loop at RL training time — it would require millions of annotations.  
The solution: train a **reward model** (RM) that learns human preferences and can score any (prompt, response) pair automatically.

The RM is the bridge between human judgment and the RL training loop:
$$\text{Human annotations} \xrightarrow{\text{train RM}} r_\phi(x, y) \xrightarrow{\text{PPO/DPO}} \pi_\theta$$

## The Bradley-Terry Preference Model

Given a preference pair $(x, y_w, y_l)$ where humans prefer $y_w$ over $y_l$, we model the preference probability as:

$$P(y_w \succ y_l \mid x) = \sigma\!\left(r_\phi(x, y_w) - r_\phi(x, y_l)\right)$$

Where:
- $r_\phi(x, y)$ is our learned scalar reward function
- $\sigma$ is the sigmoid — maps a real-valued margin to a probability
- The probability depends only on the *margin* $r_w - r_l$, not the absolute values

**Why the margin?** The reward scale is arbitrary — adding a constant $c$ to all rewards changes nothing.  
The Bradley-Terry model correctly captures this: only relative comparisons matter.

### Maximum-Likelihood Loss

Given the annotated dataset $\mathcal{D} = \{(x^{(i)}, y_w^{(i)}, y_l^{(i)})\}$, we minimise the negative log-likelihood:

$$\boxed{\mathcal{L}_{RM} = -\underset{(x,y_w,y_l) \sim \mathcal{D}}{\mathbb{E}}\!\left[\log \sigma\!\left(r_\phi(x, y_w) - r_\phi(x, y_l)\right)\right]}$$

This is binary cross-entropy on the reward margin — a well-understood, stable training objective.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Colab setup
# !pip install -q transformers datasets trl accelerate

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

## Architecture: GPT-2 with a Scalar Head

We replace GPT-2's language modelling head (which outputs a distribution over vocabulary) with a single linear layer that outputs one scalar:

```
Input: [prompt + response tokens]
     ↓
GPT-2 Transformer (all layers, causal attention)
     ↓
Last non-padding token hidden state  ← "reads" the whole sequence
     ↓
Linear(hidden_size → 1, bias=False)
     ↓
r ∈ ℝ  (scalar reward)
```

**Why the last token?** GPT-2 uses left-to-right causal attention — token $t$ can attend to all tokens $\leq t$.  
The final real token (at the end of the response) has "read" the entire sequence and provides the richest summary.  
This is analogous to using `[CLS]` in BERT, but for auto-regressive models.

In [ ]:
from src.models.reward_model import GPT2RewardModel, preference_loss

# Build the reward model from a raw GPT-2 backbone
model = GPT2RewardModel.from_pretrained_backbone('gpt2')
print('Architecture:')
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Reward head parameters: {sum(p.numel() for p in model.reward_head.parameters()):,}')

In [ ]:
# Demonstrate forward pass
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

texts = [
    'Human: Explain gravity.\n\nAssistant: Gravity is a fundamental force that attracts objects with mass toward each other.',
    'Human: Explain gravity.\n\nAssistant: I dunno lol',
]
enc = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=128)

with torch.no_grad():
    out = model(**enc)

print('Rewards:')
for text, reward in zip(texts, out.rewards):
    response = text.split('Assistant: ')[-1][:50]
    print(f'  {response!r:50s}  →  r = {reward.item():.4f}')

## The Preference Loss — Unpacked

The `preference_loss` function implements:

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^B \log \sigma(r_w^{(i)} - r_l^{(i)})$$

Along with a pairwise accuracy metric: $\text{acc} = \frac{1}{B}\sum_i \mathbf{1}[r_w^{(i)} > r_l^{(i)}]$

Random accuracy = 0.50 (chance). A well-trained RM should reach ≥0.70 on held-out pairs.

In [ ]:
# Demonstrate the preference loss
print('=== preference_loss demonstration ===')
print()

# Before training: rewards are near-random
chosen_rewards_untrained  = torch.tensor([0.12, -0.05,  0.31, -0.18])
rejected_rewards_untrained = torch.tensor([0.09, -0.01,  0.28, -0.14])
loss_untrained, acc_untrained = preference_loss(chosen_rewards_untrained, rejected_rewards_untrained)
print(f'Untrained RM:  loss={loss_untrained:.4f}  acc={acc_untrained:.2f}')
print(f'  (acc ≈ 0.5 is expected — model has not learned preferences yet)')
print()

# After training: chosen rewards reliably higher than rejected
chosen_rewards_trained  = torch.tensor([1.82, 0.95,  2.14, 1.43])
rejected_rewards_trained = torch.tensor([0.31, 0.12, -0.05, 0.88])
loss_trained, acc_trained = preference_loss(chosen_rewards_trained, rejected_rewards_trained)
print(f'Trained RM:    loss={loss_trained:.4f}  acc={acc_trained:.2f}')
print(f'  (acc ≈ 1.0 means model consistently ranks chosen > rejected)')

## Training the Reward Model

In [ ]:
from src.training.reward import RewardConfig, train_reward_model

# Demo config — needs SFT checkpoint from notebook 02
# For production: use sft_checkpoint='checkpoints/sft', num_epochs=2, fp16=True
cfg = RewardConfig(
    sft_checkpoint='checkpoints/sft_demo',
    output_dir='checkpoints/reward_model_demo',
    num_epochs=1,
    batch_size=2,
    gradient_accumulation_steps=2,
    num_train_samples=200,
    num_eval_samples=50,
    fp16=False,
    log_every=20,
)
train_reward_model(cfg)

## Training Dynamics

The key metrics to monitor during reward model training:

In [ ]:
# Representative training curves from a full 10k-sample, 2-epoch run
steps = np.arange(0, 500, 5)
rng = np.random.RandomState(0)

loss_curve = 0.693 + 0.4 * np.exp(-steps / 80) + 0.02 * rng.randn(len(steps))
acc_curve  = 0.50 + 0.22 * (1 - np.exp(-steps / 100)) + 0.01 * rng.randn(len(steps))
acc_curve  = np.clip(acc_curve, 0.49, 0.79)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(steps, loss_curve, color='tomato', linewidth=1.5, label='Preference loss')
axes[0].axhline(0.693, color='gray', linestyle='--', linewidth=1, label='Random baseline (log 2)')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Bradley-Terry preference loss')
axes[0].legend()
axes[0].set_ylim(0.3, 1.0)

axes[1].plot(steps, acc_curve, color='steelblue', linewidth=1.5, label='Pairwise accuracy')
axes[1].axhline(0.50, color='gray', linestyle='--', linewidth=1, label='Random (0.50)')
axes[1].axhline(0.70, color='seagreen', linestyle=':', linewidth=1, label='Target (0.70)')
axes[1].set_xlabel('Training step')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Pairwise accuracy (chosen vs rejected)')
axes[1].legend()
axes[1].set_ylim(0.45, 0.85)

plt.suptitle('Reward Model Training — 10k samples, 2 epochs, GPT-2 Medium', y=1.02)
plt.tight_layout()
plt.savefig('rm_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Final accuracy: {acc_curve[-1]:.3f} (target ≥ 0.70)')

## Limitations and Reward Hacking

The reward model is an *approximation* of human preferences — it has blind spots:

1. **Distribution shift**: the RM is trained on SFT responses but will score RL-generated responses.  
   If the RL policy drifts far from the training distribution, RM scores become unreliable.

2. **Length bias**: reward models tend to assign higher scores to longer responses, even if they're verbose.  
   This is mitigated by the KL penalty in PPO/DPO.

3. **Annotation artifacts**: human annotators may have consistent but non-ideal biases  
   (e.g., preferring responses that sound confident even if they're wrong).

These are the fundamental reasons we need a **KL penalty** during policy optimisation — to prevent the policy from exploiting RM blind spots.

---
**Next**: [04_ppo_training.ipynb](04_ppo_training.ipynb) — PPO policy optimisation using the reward model